# 🌍 Day 01a: HTTP & HTTPS — The Protocol Powering the Web

---

## 🎯 What You'll Master Today
- HTTP/1.1 vs HTTP/2 vs HTTP/3
- Methods, headers, status codes
- Cookies, sessions, caching
- HTTPS = HTTP + TLS

---

```
╔══════════════════════════════════════════════════════════════════════╗
║                     HTTP REQUEST / RESPONSE                         ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  CLIENT                              SERVER                          ║
║  ┌────────┐                          ┌────────┐                     ║
║  │Browser │ ──── HTTP Request ────→  │  Web   │                     ║
║  │        │ ←── HTTP Response ─────  │ Server │                     ║
║  └────────┘                          └────────┘                     ║
║                                                                      ║
║  Request:                    Response:                               ║
║  ┌──────────────────┐       ┌──────────────────┐                    ║
║  │ GET / HTTP/1.1   │       │ HTTP/1.1 200 OK  │                    ║
║  │ Host: example.com│       │ Content-Type: ... │                    ║
║  │ Accept: text/html│       │ Content-Length: 42│                    ║
║  │                  │       │                    │                    ║
║  │ [body optional]  │       │ <html>...</html>  │                    ║
║  └──────────────────┘       └──────────────────┘                    ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. HTTP Methods (Verbs)
# ============================================

methods = [
    ("GET",     "Read",    "✅", "✅", "❌", "Fetch resource"),
    ("POST",    "Create",  "❌", "❌", "✅", "Submit data / create"),
    ("PUT",     "Replace", "✅", "❌", "✅", "Full update of resource"),
    ("PATCH",   "Update",  "❌", "❌", "✅", "Partial update"),
    ("DELETE",  "Delete",  "✅", "❌", "❌", "Remove resource"),
    ("HEAD",    "Meta",    "✅", "✅", "❌", "GET without body"),
    ("OPTIONS", "Info",    "✅", "✅", "❌", "Supported methods (CORS)"),
]

print("  ╔══════════════════════════════════════════════════════════════╗")
print("  ║  Method  │ CRUD    │ Idempotent │ Safe │ Body │ Description ║")
print("  ╠══════════════════════════════════════════════════════════════╣")
for method, crud, idem, safe, body, desc in methods:
    print(f"  ║  {method:7} │ {crud:7} │     {idem}     │  {safe}  │  {body}  │ {desc:20} ║")
print("  ╚══════════════════════════════════════════════════════════════╝")

print("\n  Idempotent: same request N times = same result")
print("  Safe: doesn't modify server state (read-only)")
print("  POST is neither safe nor idempotent!")
print("\n  PUT vs PATCH:")
print("    PUT: send ENTIRE resource (replace) → {name:'Alice', age:30, city:'NYC'}")
print("    PATCH: send PARTIAL update          → {age: 31}")

In [ ]:
# ============================================
# 2. HTTP Status Codes (Must-Know)
# ============================================

status_codes = {
    "1xx Informational": [
        (100, "Continue", "Server received headers, send body"),
        (101, "Switching Protocols", "Upgrading to WebSocket"),
    ],
    "2xx Success": [
        (200, "OK", "Request succeeded"),
        (201, "Created", "Resource created (POST response)"),
        (204, "No Content", "Success but no body (DELETE response)"),
    ],
    "3xx Redirection": [
        (301, "Moved Permanently", "URL changed forever (SEO: update links)"),
        (302, "Found", "Temporary redirect (POST→GET, old behavior)"),
        (304, "Not Modified", "Use cached version (ETag match)"),
        (307, "Temporary Redirect", "Like 302 but preserves method"),
        (308, "Permanent Redirect", "Like 301 but preserves method"),
    ],
    "4xx Client Error": [
        (400, "Bad Request", "Malformed request syntax"),
        (401, "Unauthorized", "Authentication required (not logged in)"),
        (403, "Forbidden", "Authenticated but not authorized"),
        (404, "Not Found", "Resource doesn't exist"),
        (405, "Method Not Allowed", "GET on POST-only endpoint"),
        (409, "Conflict", "Resource conflict (duplicate)"),
        (429, "Too Many Requests", "Rate limited"),
    ],
    "5xx Server Error": [
        (500, "Internal Server Error", "Generic server crash"),
        (502, "Bad Gateway", "Upstream server error (proxy/LB)"),
        (503, "Service Unavailable", "Server overloaded/maintenance"),
        (504, "Gateway Timeout", "Upstream didn't respond in time"),
    ],
}

print("  ═══ HTTP STATUS CODES ═══\n")
for category, codes in status_codes.items():
    print(f"  {category}:")
    for code, name, desc in codes:
        print(f"    {code} {name:25} → {desc}")
    print()

print("  💼 Interview trick: 401 vs 403?")
print("    401: WHO are you? (not authenticated)")
print("    403: I know who you are, but NO. (not authorized)")

In [ ]:
# ============================================
# 3. HTTP Versions Comparison
# ============================================

print("""
  ┌──────────────┬──────────────────┬──────────────────┬──────────────────┐
  │ Feature       │ HTTP/1.1         │ HTTP/2           │ HTTP/3           │
  ├──────────────┼──────────────────┼──────────────────┼──────────────────┤
  │ Year          │ 1997             │ 2015             │ 2022             │
  │ Transport     │ TCP              │ TCP              │ QUIC (UDP)       │
  │ Format        │ Text             │ Binary           │ Binary           │
  │ Multiplexing  │ ❌ (1 req/conn)  │ ✅ (streams)     │ ✅ (streams)     │
  │ Head-of-line  │ ❌ Blocking      │ ⚠️ TCP level     │ ✅ Fixed (QUIC)  │
  │ Header comp.  │ ❌ None          │ ✅ HPACK         │ ✅ QPACK         │
  │ Server Push   │ ❌               │ ✅               │ ✅               │
  │ TLS required  │ Optional         │ Practically yes  │ Built-in (1-RTT) │
  │ Connection    │ Keep-Alive       │ Single conn      │ 0-RTT possible   │
  └──────────────┴──────────────────┴──────────────────┴──────────────────┘

  HTTP/1.1 problem: Head-of-line blocking
    Request 1 ──[waiting]──→ Response 1
                             Request 2 ──[waiting]──→ Response 2  (blocked!)
    Workaround: browsers open 6 parallel TCP connections

  HTTP/2 solution: Multiplexing (multiple streams on 1 connection)
    ──[Req1]──[Req2]──[Req3]──→   (all on same connection)
    ←──[Resp2]──[Resp1]──[Resp3]   (responses can arrive in any order)

  HTTP/3 innovation: QUIC over UDP
    - No TCP head-of-line blocking (lost packet only affects its stream)
    - 0-RTT handshake (for repeat connections)
    - Built-in encryption (TLS 1.3 integrated)
""")

In [ ]:
# ============================================
# 4. Cookies, Sessions, Caching
# ============================================

print("""
  ═══ COOKIES & SESSIONS ═══

  Cookie: Server → Client    (Set-Cookie header)
  ┌──────────┐  Set-Cookie: session_id=abc123  ┌──────────┐
  │  Client  │ ←────────────────────────────── │  Server  │
  │          │ ──────────────────────────────→ │          │
  └──────────┘  Cookie: session_id=abc123       └──────────┘
                (sent with every subsequent request)

  Cookie attributes:
    Domain, Path, Expires/Max-Age, Secure, HttpOnly, SameSite

  HttpOnly: JavaScript CANNOT access (prevents XSS theft)
  Secure:   Only sent over HTTPS
  SameSite: Strict/Lax/None (CSRF protection)
""")

print("""
  ═══ HTTP CACHING ═══

  Headers:
    Cache-Control: max-age=3600         → cache for 1 hour
    Cache-Control: no-cache             → always revalidate
    Cache-Control: no-store             → never cache (sensitive data)
    ETag: "abc123"                      → content hash (fingerprint)
    If-None-Match: "abc123"             → client sends to revalidate
    → 304 Not Modified (use cache)  OR  200 OK (new content)

  Flow:
    1st request: Server → 200 OK + ETag: "v1"
    2nd request: Client → If-None-Match: "v1"
    If unchanged: Server → 304 Not Modified (no body — saves bandwidth!)
    If changed:   Server → 200 OK + ETag: "v2" + new body
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | GET vs POST? | GET: idempotent, cacheable, params in URL. POST: non-idempotent, body, creates resources |
| 2 | 301 vs 302? | 301: permanent redirect (update bookmarks/SEO). 302: temporary redirect |
| 3 | HTTP/1.1 vs HTTP/2? | H2: binary, multiplexed streams, header compression, server push, single connection |
| 4 | What is HTTPS? | HTTP + TLS encryption. Port 443. Provides confidentiality, integrity, authentication |
| 5 | Cookie vs Session? | Cookie: client-side storage. Session: server-side state, referenced by session_id cookie |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • HTTP is stateless, text-based request/response        │
## │  • Know ALL status codes (200, 201, 301, 400, 401, 403)  │
## │  • HTTP/2: multiplexing, binary, HPACK compression       │
## │  • HTTP/3: QUIC (UDP), no HOL blocking, 0-RTT            │
## │  • Cookies: HttpOnly + Secure + SameSite for security    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **DNS** — How domain names become IP addresses